In [3]:
from langchain_core.documents import Document

In [4]:
doc = Document(
    page_content="Hello this is girish",
    metadata={
        "name":"girish",
        "subject":"dsa"
    }
)
doc

Document(metadata={'name': 'girish', 'subject': 'dsa'}, page_content='Hello this is girish')

In [ ]:
# Vector DB pipeline

In [5]:
from langchain_community.document_loaders import TextLoader

loder = TextLoader("./data/text_files/file1.txt",encoding="utf-8")
loder.load()

d:\Coding\All Codes\Working on\RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [4]:
from langchain_community.document_loaders import DirectoryLoader

dir_loader = DirectoryLoader(
    "./data/text_files",
    glob="**/*txt",
    loader_cls=TextLoader,
    loader_kwargs={'encoding':'utf-8'},
    show_progress=False
)
dir_loader.load()


[Document(metadata={'source': 'data\\text_files\\file1.txt'}, page_content='HI I am founder of GTcoder foundation.\nwhich help many engineers to learn CP.'),
 Document(metadata={'source': 'data\\text_files\\IMP Websites.txt'}, page_content='1] for all cheetsheets  ->  https://overapi.com/\n\t\n\n\n')]

In [33]:
from langchain_community.document_loaders import PyPDFLoader, PyMuPDFLoader


dir_loader = DirectoryLoader(
    "./data/pdfs",
    glob="**/*pdf",
    loader_cls=PyPDFLoader,
    show_progress=False
)
chunks = dir_loader.load()

In [6]:
import numpy as np
from sentence_transformers import SentenceTransformer
from chromadb.config import Settings
import chromadb
import uuid
import os
from typing import List, Dict, Any, Tuple
from sklearn.metrics.pairwise import cosine_similarity

In [7]:
class EmbeddingManager:
    # """Handles document embedding generation using SentenceTransformer"""

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        # Initialize the embedding manager
        self.model_name = model_name
        self.model = None
        self._load_model()

    def _load_model(self):
        # """Load the SentenceTransformer model"""
        try:
            print(f"Loading embedding model: {self.model_name}")
            self.model = SentenceTransformer(self.model_name)
            print(f"Model loaded successfully. Embedding dimension: {self.model.get_embedding_dimension()}")
        except Exception as e:
            print(f"Error loading model {self.model_name}: {e}")
            raise

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        # Generate embeddings for a list of texts
        # Args :
        # texts: List of text strings to embed
        # Returns:
        # numpy array of embeddings with shape (len(texts), embedding_dim)
    
        if not self.model:
            raise ValueError("Model not loaded")

        print(f"Generating embeddings for {len(texts)} texts ... ")
        embeddings = self.model.encode(texts, show_progress_bar=True)
        print(f"Generated embeddings with shape: {embeddings.shape}")
        return embeddings

embedding_manager=EmbeddingManager()
embedding_manager

Loading embedding model: all-MiniLM-L6-v2


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2661.44it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model loaded successfully. Embedding dimension: 384


In [9]:
class VectorStore:
# """Manages document embeddings in a ChromaDB vector store"""

    def __init__(self, collection_name: str = "pdf_documents", persist_directory: str = "./data/vector_store"):
        # Args:
        # collection_name: Name of the ChromaDB collection
        # persist_directory: Directory to persist the vector store

        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_store()

    def _initialize_store(self):
        # """Initialize ChromaDB client and collection"""
        try:
            # Create persistent ChromaDB client
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(path=self.persist_directory)

            # Get or create collection
            self.collection = self.client.get_or_create_collection(
                name=self.collection_name,
                metadata={"description": "PDF document embeddings for RAG"}
            )
            print(f"Vector store initialized. Collection: {self.collection_name}")
            print(f"Existing documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error initializing vector store: {e}")
            raise
    
    def add_documents(self, documents: List[Any], embeddings: np.ndarray):
        # Add documents and their embeddings to the vector store
        # Args:
        # documents: List of LangChain documents
        # embeddings: Corresponding embeddings for the documents

        if len(documents) != len(embeddings):
            raise ValueError("Number of documents must match number of embeddings")

        print(f"Adding {len(documents)} documents to vector store ... ")

        # Prepare data for ChromaDB
        ids = []
        metadatas = []
        documents_text = []
        embeddings_list = []


        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            # Generate unique ID
            doc_id = f"doc_{uuid.uuid4().hex[:8]}_{i}"
            ids.append(doc_id)

            # Prepare metadata
            metadata = dict(doc.metadata)
            metadata['doc_index'] = i
            metadata['content_length'] = len(doc.page_content)
            metadatas.append(metadata)

            # Document content
            documents_text.append(doc.page_content)

            # Embedding
            embeddings_list.append(embedding.tolist())

        # Add to collection
        try:
            self.collection.add(
                ids=ids,
                embeddings=embeddings_list,
                metadatas=metadatas,
                documents=documents_text
            )
            print(f"Successfully added {len(documents)} documents to vector store")
            print(f"Total documents in collection: {self.collection.count()}")

        except Exception as e:
            print(f"Error adding documents to vector store: {e}")
            raise

vector_store=VectorStore()
vector_store 

Vector store initialized. Collection: pdf_documents
Existing documents in collection: 10


In [35]:
chunks

[Document(metadata={'producer': 'Skia/PDF m134 Google Docs Renderer', 'creator': 'PyPDF', 'creationdate': '', 'title': 'Docker CheatSheet ApnaCollege', 'source': 'data\\pdfs\\Docker CheatSheet ApnaCollege.pdf', 'total_pages': 4, 'page': 0, 'page_label': '1'}, page_content='Docker  Commands    IMAGES  :   List  all  Local  images  docker  images    Delete  an  image  docker  rmi  <image_name>   Remove  unused  images  docker  image  prune    Build  an  image  from  a  Dockerﬁle  docker  build  -t  <image_name>:<version>  .    //version  is  optional   docker  build  -t  <image_name>:<version>  .  -no-cache  //build  without  cache     CONTAINER :  List  all  Local  containers  (running  &  stopped)  docker  ps  -a   List  all  running  containers  docker  ps  Create  &  run  a  new  container  docker  run  <image_name>   //if  image  not  available  locally,  it’ll  be  downloaded  from  DockerHub  Run  container  in  background  docker  run  -d  <image_name>    Run  container  with  cu

In [36]:
# converting text to embedding
texts = [doc.page_content for doc in chunks]
texts

['Docker  Commands    IMAGES  :   List  all  Local  images  docker  images    Delete  an  image  docker  rmi  <image_name>   Remove  unused  images  docker  image  prune    Build  an  image  from  a  Dockerﬁle  docker  build  -t  <image_name>:<version>  .    //version  is  optional   docker  build  -t  <image_name>:<version>  .  -no-cache  //build  without  cache     CONTAINER :  List  all  Local  containers  (running  &  stopped)  docker  ps  -a   List  all  running  containers  docker  ps  Create  &  run  a  new  container  docker  run  <image_name>   //if  image  not  available  locally,  it’ll  be  downloaded  from  DockerHub  Run  container  in  background  docker  run  -d  <image_name>    Run  container  with  custom  name  docker  run  -  -name  <container_name>  <image_name>',
 'Port  Binding  in  container  docker  run  -p<host_port>:<container_port>  <image_name>    Set  environment  variables  in  a  container  docker  run  -e   <var_name>=<var_value>   <container_name>     

In [41]:
#  Generate the embeddings
embeddings = embedding_manager.generate_embeddings(texts)
embeddings
vector_store.add_documents(chunks,embeddings)

Generating embeddings for 10 texts ... 


Batches: 100%|██████████| 1/1 [00:00<00:00,  2.85it/s]

Generated embeddings with shape: (10, 384)
Adding 10 documents to vector store ... 
Successfully added 10 documents to vector store
Total documents in collection: 10


In [ ]:
# RAG Retrieval pipeline 

In [10]:
class RAGRetriever():
    """Handles query-based retrieval from the vector store"""

    def __init__(self, vector_store: VectorStore, embedding_manager: EmbeddingManager) :
        # Args :
        # vector_store: Vector store containing document embeddings
        # embedding_manager: Manager for generating query embeddings
        self.vector_store = vector_store
        self.embedding_manager = embedding_manager

    def retrieve(self, query: str, top_k: int = 5, score_threshold: float = 0.0) -> List[Dict[str, Any]]:
        # Retrieve relevant documents for a query
        # Args :
        # query: The search query
        # top_k: Number of top results to return
        # score threshold: Minimum similarity score threshold
        # Returns:
        # List of dictionaries containing retrieved documents and metadata
        print(f"Retrieving documents for query: '{query}'")
        print(f"Top K: {top_k}, Score threshold: {score_threshold}")

        # Generate query embedding
        query_embedding = self.embedding_manager.generate_embeddings([query])[0]

        # Search in vector store
        try:
            results = self.vector_store.collection.query(
                query_embeddings=[query_embedding.tolist()],
                n_results=top_k
            )

            # Process results
            retrieved_docs = []

            if results['documents'] and results['documents'][0]:
                documents = results['documents'][0]
                metadatas = results['metadatas'][0]
                distances = results['distances'][0]
                ids = results['ids'][0]

                for i, (doc_id, document, metadata, distance) in enumerate(zip(ids, documents, metadatas, distances)):
                    # Convert distance to similarity score (ChromaDB uses cosine distance)
                    similarity_score = 1 - distance

                    if similarity_score >= score_threshold:
                        retrieved_docs.append({
                            'id': doc_id,
                            'content': document,
                            'metadata': metadata,
                            'similarity_score': similarity_score,
                            'distance' : distance,
                            'rank': i + 1
                        })

                print(f"Retrieved {len(retrieved_docs)} documents (after filtering)")
            else:
                print("No Documents found")
            return retrieved_docs
    
        except Exception as e:
            print("Error During retrieval")
            return []

rag_retriever = RAGRetriever(vector_store,embedding_manager)



In [48]:
rag_retriever

In [12]:
res = rag_retriever.retrieve("docker make new container ")
res 

Retrieving documents for query: 'docker make new container '
Top K: 5, Score threshold: 0.0
Generating embeddings for 1 texts ... 


Batches: 100%|██████████| 1/1 [00:00<00:00, 35.06it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


[{'id': 'doc_e58824eb_0',
  'content': 'Docker  Commands    IMAGES  :   List  all  Local  images  docker  images    Delete  an  image  docker  rmi  <image_name>   Remove  unused  images  docker  image  prune    Build  an  image  from  a  Dockerﬁle  docker  build  -t  <image_name>:<version>  .    //version  is  optional   docker  build  -t  <image_name>:<version>  .  -no-cache  //build  without  cache     CONTAINER :  List  all  Local  containers  (running  &  stopped)  docker  ps  -a   List  all  running  containers  docker  ps  Create  &  run  a  new  container  docker  run  <image_name>   //if  image  not  available  locally,  it’ll  be  downloaded  from  DockerHub  Run  container  in  background  docker  run  -d  <image_name>    Run  container  with  custom  name  docker  run  -  -name  <container_name>  <image_name>',
  'metadata': {'content_length': 791,
   'source': 'data\\pdfs\\Docker CheatSheet ApnaCollege.pdf',
   'title': 'Docker CheatSheet ApnaCollege',
   'doc_index': 0,
  

In [13]:
from langchain_groq import ChatGroq
import os
from dotenv import load_dotenv 
load_dotenv()



True

In [ ]:

# groq_api_key = os.getenv("GROQ_API_KEY")

In [18]:
llm = ChatGroq(api_key=groq_api_key,model="llama-3.3-70b-versatile",temperature=0.1,max_tokens=1024)


def rag_simple(query,retriever,llm,top_k=3):

    results=retriever.retrieve(query, top_k=top_k)
    context="\n\n".join([doc['content'] for doc in results]) if results else ""
    if not context:
        return "No relevant context found to answer the question."

    prompt=f"""Use the following context to answer the question concisely.
        Context:
        {context}

        Question: {query}

        Answer:
    """
    response=llm.invoke([prompt.format(context=context, query=query)])
    return response.content

In [20]:
answer = rag_simple("docker make new container",rag_retriever,llm)
answer

Retrieving documents for query: 'docker make new container'
Top K: 3, Score threshold: 0.0
Generating embeddings for 1 texts ... 


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00, 18.93it/s]

Generated embeddings with shape: (1, 384)
Retrieved 3 documents (after filtering)


'docker run <image_name> \n\n(This command will create and run a new container from the specified image. If the image is not available locally, it will be downloaded from DockerHub.)'

In [21]:
def rag_advanced(query, retriever, llm, top_k=5, min_score=0.2, return_context=False):

    # RAG pipeline with extra features:
    # - Returns answer, sources, confidence score, and optionally

    results = retriever.retrieve(query, top_k=top_k, score_threshold=min_score)
    if not results:
        return {'answer': 'No relevant context found.', 'sources': [], 'confidence' : 0.0, 'context': ''}

    # Prepare context and sources
    context = "\n\n".join([doc['content'] for doc in results])
    sources = [{
        'source' : doc['metadata' ].get('source_file' , doc['metadata' ].get('source', 'unknown' ) ),
        'page' : doc['metadata' ].get('page', 'unknown' ),
        'score' : doc['similarity_score' ],
        'preview': doc['content' ] [ : 120] + '...'
        } for doc in results]
    confidence = max([doc['similarity_score' ] for doc in results])

    # Generate answer
    prompt = f"""Use the following context to answer the question concisely. \nContext: \n{context}\n\nQuestion: {query}\n\nAnswer"""
    response = llm.invoke([prompt.format(context=context, query=query)])

    output = {
        'answer': response.content,
        'sources': sources,
        'confidence': confidence
    }

    if return_context:
        output['context'] = context
    return output


# Example usage:
result = rag_advanced("how to make docker images ?",rag_retriever, llm, top_k=3, min_score=0.1, return_context=True)
print("Answer:", result['answer'])
print("Sources:", result['sources'])
print("Confidence:", result['confidence'])
print("Context Preview:", result['context'][:300])



Retrieving documents for query: 'how to make docker images ?'
Top K: 3, Score threshold: 0.1
Generating embeddings for 1 texts ... 


Batches: 100%|██████████| 1/1 [00:00<00:00,  4.29it/s]


Generated embeddings with shape: (1, 384)
Retrieved 2 documents (after filtering)
Answer: To make a Docker image, use the command: `docker build -t <image_name>:<version> .` 

(Note: `<version>` is optional and `.` represents the current directory where the Dockerfile is located)
Sources: [{'source': 'data\\pdfs\\Docker CheatSheet ApnaCollege.pdf', 'page': 0, 'score': 0.3476353883743286, 'preview': 'Docker  Commands    IMAGES  :   List  all  Local  images  docker  images    Delete  an  image  docker  rmi  <image_name>...'}, {'source': 'data\\pdfs\\Docker CheatSheet ApnaCollege.pdf', 'page': 1, 'score': 0.13906848430633545, 'preview': 'Port  Binding  in  container  docker  run  -p<host_port>:<container_port>  <image_name>    Set  environment  variables  ...'}]
Confidence: 0.3476353883743286
Context Preview: Docker  Commands    IMAGES  :   List  all  Local  images  docker  images    Delete  an  image  docker  rmi  <image_name>   Remove  unused  images  docker  image  prune    Build  an  